In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH="homo.pdf" 

loader=PyPDFLoader(PDF_PATH)
pages=loader.load()

print(f"Loaded {len(pages)} pages from PDF")
print("\n---First page preview (First 500 chars)---")
print(pages[0].page_content[:500])

C:\Users\kumbh\AppData\Local\Temp\ipykernel_39672\425046800.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
D:\tutorial-agentic-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 26 pages from PDF

---First page preview (First 500 chars)---
RESEARCH Open Access
© The Author(s) 2025. Open Access  This article is licensed under a Creative Commons Attribution-NonCommercial-NoDerivatives 4.0 
International License, which permits any non-commercial use, sharing, distribution and reproduction in any medium or format, as long as you 
give appropriate credit to the original author(s) and the source, provide a link to the Creative Commons licence, and indicate if you modified the 
licensed material. You do not have permission under this lic


In [3]:
print(pages[5].page_content[:500])

Page 6 of 26
Junior et al. Journal of Cloud Computing            (2025) 14:84 
such as dynamic allocation to meet real-time demand 
[6], virtualization to improve resource utilization  utili -
zation [ 80], and energy-efficient models like Decentral -
ized Energy-Aware Collaborative Model (DEACM) for 
minimizing consumption [ 80]. Scheduling techniques 
enhance performance, with task scheduling and load 
balancing preventing bottlenecks and improving consis -
tency [6, 67]. Challenges involve sc


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n","\n","."," "]
)

chunk=splitter.split_documents(pages)
len(chunk)

137

In [10]:
chunk[0]


Document(metadata={'producer': 'Adobe PDF Library 17.0; modified using iText® 5.5.13.4 ©2000-2024 iText Group NV (SPRINGER SBM; licensed version)', 'creator': 'Adobe InDesign 18.0 (Windows)', 'creationdate': '2025-11-21T03:16:24+01:00', 'crossmarkdomains[1]': 'springer.com', 'crossmarkdomains[2]': 'springerlink.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'moddate': '2025-12-30T00:02:36+01:00', 'trapped': '/False', 'doi': '10.1186/s13677-025-00774-5', 'source': 'homo.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}, page_content='RESEARCH Open Access\n© The Author(s) 2025. Open Access  This article is licensed under a Creative Commons Attribution-NonCommercial-NoDerivatives 4.0 \nInternational License, which permits any non-commercial use, sharing, distribution and reproduction in any medium or format, as long as you \ngive appropriate credit to the original author(s) and the source, provide a link to the Creative Commons licence, and indic

In [11]:
chunk[0].page_content


'RESEARCH Open Access\n© The Author(s) 2025. Open Access  This article is licensed under a Creative Commons Attribution-NonCommercial-NoDerivatives 4.0 \nInternational License, which permits any non-commercial use, sharing, distribution and reproduction in any medium or format, as long as you \ngive appropriate credit to the original author(s) and the source, provide a link to the Creative Commons licence, and indicate if you modified the \nlicensed material. You do not have permission under this licence to share adapted material derived from this article or parts of it. The images or \nother third party material in this article are included in the article’s Creative Commons licence, unless indicated otherwise in a credit line to the \nmaterial. If material is not included in the article’s Creative Commons licence and your intended use is not permitted by statutory regulation or'

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store=Chroma.from_documents(chunk,embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8615.45it/s]


Vector store ready. 274 vectors stored.


In [30]:
retriever=vector_store.as_retriever(search_kwargs={"k":3})
test_query="What is Homomorphic Encryption?"
retrieved=retriever.invoke(test_query)

for i,doc in enumerate(retrieved,1):
    print(f"---Chunk{i}---")
    print(doc.page_content[:300])
    print()

---Chunk1---
[39]. Techniques like fuzzy keyword search and finger -
print identification secure access to encrypted files [ 16]. 
Key management and evolving threats remain major 
challenges to cryptographic security. Block ciphers 
encrypt data in fixed-size blocks and resist known attacks 
[28]. Stream cipher

---Chunk2---
[39]. Techniques like fuzzy keyword search and finger -
print identification secure access to encrypted files [ 16]. 
Key management and evolving threats remain major 
challenges to cryptographic security. Block ciphers 
encrypt data in fixed-size blocks and resist known attacks 
[28]. Stream cipher

---Chunk3---
data privacy protection. It is was followed by the usage 
of Partially Homomorphic algorithm, which represented 
24 articles of the total 75. 11 articles did not specifically 
state the type of homomorphic algorithm use to protect 
the cloud data even though it categorically mentioned the 
use of Ho



In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq 

#--- Helper:join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n--\n\n".join(doc.page_content for doc in docs)
    
SYSTEM_PROMPT="""\
you are helpful cloud computation and encryption assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information,say so clearly

Context:
{context}
"""

prompt=ChatPromptTemplate.from_messages([
    ("system",SYSTEM_PROMPT),
    ("human","{question}")
])

#---llm via Groq API---
llm=ChatGroq(model="qwen/qwen3-32b",
            temperature=0,
            reasoning_format="parsed",
            )

chain=(
    {"context":retriever |format_docs,"question":RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

print("RAG chain Assembled")

RAG chain Assembled


In [27]:
question="what is meant by homomorphic encrytion?"

print(f"Q:{question}\n")
print("A:",chain.invoke(question))

Q:what is meant by homomorphic encrytion?

A: Homomorphic encryption (HE) is a cryptographic technique that allows computations to be performed directly on encrypted data without requiring decryption first. The results of these computations remain encrypted, and when decrypted, they match the results of operations performed on the original plaintext. This enables secure data processing in untrusted environments (e.g., cloud computing) while maintaining data confidentiality throughout the computation. 

Homomorphic encryption schemes are categorized by the operations they support, such as **Partially Homomorphic Encryption (PHE)**, which supports specific types of computations. The context highlights its role in addressing data privacy challenges in cloud environments, as noted in recent research trends (2022–2024).
